In [1]:
import kagglehub
import torch
from torch import nn
import pandas as pd
import torchinfo

/root/autodl-tmp/envs/daily_learning/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/root/autodl-tmp/envs/daily_learning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


获取数据


In [2]:
# Download latest version
try:
    path = kagglehub.competition_download('playground-series-s6e8', output_dir='./data')
    print("Path to competition files:", path)
except Exception as e:
    print("Error downloading competition files:", e)

Path to competition files: ./data


查看数据


In [3]:
train_data = pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test.csv')
train_data.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


字符串特征处理


In [4]:
dum_cols = ['gender', 'stress_level', 'academic_work_impact']
train_data = pd.get_dummies(train_data, columns=dum_cols, dummy_na=True, dtype=int)
test_data = pd.get_dummies(test_data, columns=dum_cols, dummy_na=True, dtype=int)

缺失值处理


In [5]:
train_data = train_data[train_data['addicted_label'].notna()]
def fill(s: pd.Series, mean:pd.Series | None = None):
    return s.fillna(mean[s.name]) if mean is not None else s.fillna(s.mean())
train_data = train_data.apply(fill)
test_data = test_data.apply(fill, mean=train_data.mean())

分离训练标签


In [6]:
train_features = train_data.drop(columns=['id', 'addicted_label'])
train_labels = train_data['addicted_label']

归一化

In [7]:
mean = train_features.mean()
std = train_features.std()
eps = 1e-8
train_features = (train_features - mean) / (std + eps)
test_features = (test_data - mean) / (std + eps)

数据转换器


In [8]:
def get_train_loader(features: pd.DataFrame, labels: pd.Series, batch_size: int = 32):
    dataset = torch.utils.data.TensorDataset(
        torch.tensor(features.values, dtype=torch.float32),
        torch.tensor(labels.values, dtype=torch.float32)
    )
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

划分

In [44]:
val_features = train_features.sample(frac=0.2)
val_labels = train_labels[val_features.index]
train_features = train_features.drop(val_features.index)
train_labels = train_labels.drop(val_features.index)

模型


In [39]:
class Residual(nn.Module):
    def __init__(self, size):
        super(Residual, self).__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x
        x = self.relu(self.bn(self.fc1(x)))
        x = self.bn(self.fc2(x))
        x = self.relu(x + res)
        return x

class Net(nn.Module):
    def __init__(self, input_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 256)
        self.fc3 = nn.Linear(256, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x


In [40]:
torchinfo.summary(Net(len(train_features.columns)), input_size=(1, len(train_features.columns)), depth=2)

Layer (type:depth-idx)                   Output Shape              Param #
Net                                      [1, 1]                    --
├─Linear: 1-1                            [1, 64]                   1,344
├─ReLU: 1-2                              [1, 64]                   --
├─Linear: 1-3                            [1, 256]                  16,640
├─ReLU: 1-4                              [1, 256]                  --
├─Linear: 1-5                            [1, 32]                   8,224
├─ReLU: 1-6                              [1, 32]                   --
├─Linear: 1-7                            [1, 1]                    33
├─Sigmoid: 1-8                           [1, 1]                    --
Total params: 26,241
Trainable params: 26,241
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.03
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.10
Estimated Total Size (MB): 0.11

训练


In [41]:
def accuracy(net: nn.Module,
             dataloader: torch.utils.data.DataLoader,
             device: torch.device = torch.device('cpu')):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = net(inputs)
            predicted = (outputs.squeeze() > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / (total+1e-8)

In [49]:
def train(net: nn.Module,
          train_loader: torch.utils.data.DataLoader,
          valid_loader: torch.utils.data.DataLoader,
          epochs: int = 10,
          lr: float = 0.01,
          device: torch.device = torch.device('cpu'),
          verbose: bool = False):
    net.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for epoch in range(epochs):
        count = 0
        net.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            count += 1
            if verbose and count % 100 == 0:
                val_acc = accuracy(net, valid_loader, device=device)
                print(f'Epoch {epoch + 1}, Batch {count}, Batch Loss: {loss.item():.4f} Validation Accuracy: {val_acc:.4f}')
        val_acc = accuracy(net, valid_loader, device)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}, Validation Accuracy: {val_acc:.4f}')

In [56]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lr = 0.005
num_epochs = 10
batch_size = 1428
net = Net(len(train_features.columns))
train_loader = get_train_loader(train_features, train_labels, batch_size=batch_size)
valid_loader = get_train_loader(val_features, val_labels, batch_size=batch_size)
train(net, train_loader, valid_loader, epochs=num_epochs, lr=lr, device=device, verbose=True)

Epoch 1, Batch 100, Batch Loss: 0.3108 Validation Accuracy: 0.8439
Epoch 1, Batch 200, Batch Loss: 0.2964 Validation Accuracy: 0.8434
Epoch 1/10, Loss: 0.3260, Validation Accuracy: 0.8444
Epoch 2, Batch 100, Batch Loss: 0.3071 Validation Accuracy: 0.8487
Epoch 2, Batch 200, Batch Loss: 0.2972 Validation Accuracy: 0.8553
Epoch 2/10, Loss: 0.3006, Validation Accuracy: 0.8560
Epoch 3, Batch 100, Batch Loss: 0.3003 Validation Accuracy: 0.8561
Epoch 3, Batch 200, Batch Loss: 0.2924 Validation Accuracy: 0.8575
Epoch 3/10, Loss: 0.2928, Validation Accuracy: 0.8579
Epoch 4, Batch 100, Batch Loss: 0.2990 Validation Accuracy: 0.8591
Epoch 4, Batch 200, Batch Loss: 0.3091 Validation Accuracy: 0.8584
Epoch 4/10, Loss: 0.2897, Validation Accuracy: 0.8599
Epoch 5, Batch 100, Batch Loss: 0.2863 Validation Accuracy: 0.8599
Epoch 5, Batch 200, Batch Loss: 0.2998 Validation Accuracy: 0.8591
Epoch 5/10, Loss: 0.2883, Validation Accuracy: 0.8596
Epoch 6, Batch 100, Batch Loss: 0.2837 Validation Accuracy: 

预测

In [57]:
def predict(net: nn.Module,
            features: pd.DataFrame,
            device: torch.device = torch.device('cpu')):
    net.eval()
    with torch.no_grad():
        inputs = torch.tensor(features.values, dtype=torch.float32).to(device)
        outputs = net(inputs)
        # 保留概率形式
        predicted = outputs.squeeze().cpu().numpy()
    return predicted

In [58]:
predictions = predict(net, test_features.drop(columns=['id']), device=device)
submission = pd.DataFrame({'id': test_data['id'], 'addicted_label': predictions})
submission.to_csv('./data/submission.csv', index=False)